# Pneumonia Detection in Chest X-Rays with Deep Learning and Random Forest
This notebook compares three approaches for automated pneumonia detection:
- A convolutional neural network (CNN) trained from scratch.
- A Random Forest classifier on features extracted by MobileNetV2.
- MobileNetV2 with fine-tuning and a dense classification head (full transfer learning).

## 1. Import required libraries

All libraries needed for image processing, model building and training with Keras, feature extraction with MobileNetV2, classical classification with scikit-learn, and visualisation with matplotlib and seaborn.


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

## 2. Configuration: paths and global parameters

Paths to the train, validation and test splits, together with global parameters such as image size, batch size and number of training epochs.


In [ ]:
data_dir = 'data'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'validation')
test_dir = os.path.join(data_dir, 'test')
img_width, img_height = 150, 150
batch_size = 32
epochs = 10

## 3. Data generators

Generators load images from disk and process them automatically.  
On-the-fly data augmentation is applied during training to improve generalisation, and pixel values are normalised to [0, 1] by dividing by 255.  
Shuffling is disabled (`shuffle=False`) in the training and test generators to preserve label–sample correspondence when extracting MobileNetV2 features.


In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255, shear_range=0.2, zoom_range=0.2, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(train_dir, target_size=(img_width, img_height),
    batch_size=batch_size, class_mode='binary', shuffle=False)
val_generator = val_datagen.flow_from_directory(val_dir, target_size=(img_width, img_height),
    batch_size=batch_size, class_mode='binary')
test_generator = val_datagen.flow_from_directory(test_dir, target_size=(img_width, img_height),
    batch_size=1, class_mode='binary', shuffle=False)

## 4. Model 1 — Custom CNN

If `modelo_neumonia.h5` already exists it is loaded directly.  
Otherwise, a sequential CNN with three convolutional blocks is built from scratch, trained, and saved for future use.


In [ ]:
if os.path.exists('modelo_neumonia.h5'):
    print('Loading existing CNN model...')
    model = load_model('modelo_neumonia.h5')
else:
    print('Training CNN model from scratch...')
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(img_width, img_height, 3)),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Conv2D(128, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(0.0001), loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(train_generator, steps_per_epoch=train_generator.samples // batch_size,
        validation_data=val_generator, validation_steps=val_generator.samples // batch_size,
        epochs=epochs)
    model.save('modelo_neumonia.h5')

## 5. CNN predictions on the test set

Predictions are generated for all test images and converted to binary labels (0 or 1) using a 0.5 threshold.


In [ ]:
predictions = model.predict(test_generator)
predicted_classes = (predictions > 0.5).astype(int)

## 6. Feature extraction with MobileNetV2

MobileNetV2 pre-trained on ImageNet is used as a fixed feature extractor (no fine-tuning).  
Each image is mapped to a 1,280-dimensional vector that captures high-level visual patterns learned on the large ImageNet corpus.  
The extractor is saved so that the Streamlit app can reuse it at inference time for the Random Forest pipeline.


In [ ]:
mobilenet = MobileNetV2(input_shape=(img_width, img_height, 3), include_top=False,
                          weights='imagenet', pooling='avg')
mobilenet.save('modelo_mobilenet_features.h5')

train_features = mobilenet.predict(train_generator)
test_features = mobilenet.predict(test_generator)

## 7. Model 2 — MobileNetV2 + Random Forest

A `RandomForestClassifier` is trained on the feature vectors produced by MobileNetV2.  
`class_weight='balanced'` compensates for the class imbalance present in the dataset.  
The trained classifier is saved and then used to predict on the test set.


In [ ]:
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(train_features, train_generator.classes)
y_pred_rf = rf.predict(test_features)

joblib.dump(rf, 'modelo_rf.pkl')

## 8. Model 3 — MobileNetV2 Fine-tuned

Third approach: MobileNetV2 is used as a backbone, extended with a dense classification head, and trained end-to-end.

- **Phase 1:** The MobileNetV2 base is frozen. Only the custom head (Dense + Dropout) is trained for 10 epochs.
- **Phase 2:** The last 20 layers are unfrozen and the full network is fine-tuned with a reduced learning rate (`1e-5`) to avoid destroying the ImageNet representations.

Unlike Model 2, the final classifier is also a neural network rather than a classical algorithm.

In [ ]:
if os.path.exists('modelo_mobilenet_finetuned.h5'):
    print('Loading existing fine-tuned MobileNetV2 model...')
    model_ft = load_model('modelo_mobilenet_finetuned.h5')
else:
    print('Phase 1: training dense head (frozen base)...')
    base = MobileNetV2(input_shape=(img_width, img_height, 3), include_top=False, weights='imagenet')
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    model_ft = Model(inputs=base.input, outputs=output)
    model_ft.compile(optimizer=Adam(0.0001), loss='binary_crossentropy', metrics=['accuracy'])
    model_ft.fit(train_generator, steps_per_epoch=train_generator.samples // batch_size,
        validation_data=val_generator, validation_steps=val_generator.samples // batch_size,
        epochs=epochs)

    print('Phase 2: fine-tuning last 20 layers...')
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    model_ft.compile(optimizer=Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
    model_ft.fit(train_generator, steps_per_epoch=train_generator.samples // batch_size,
        validation_data=val_generator, validation_steps=val_generator.samples // batch_size,
        epochs=5)
    model_ft.save('modelo_mobilenet_finetuned.h5')

preds_ft = model_ft.predict(test_generator)
y_pred_ft = (preds_ft > 0.5).astype(int).flatten()
y_true = test_generator.classes
print('MobileNetV2 fine-tuned predictions complete.')

## 9. Comparative results across the three models

Performance metrics are computed for all three approaches:
- Overall accuracy
- **Precision**: of all cases flagged as pneumonia, how many were truly pneumonia?
- **Recall**: of all true pneumonia cases, how many did we detect?
- F1-score (harmonic mean of precision and recall)

In a medical context **recall is the most critical metric**: missing a true pneumonia case (false negative) carries a higher clinical cost than a false alarm (false positive).

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy":  round(float(accuracy_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "recall":    round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "f1":        round(float(f1_score(y_true, y_pred, zero_division=0)), 4)
    }

metrics = {
    "CNN":                    compute_metrics(y_true, predicted_classes),
    "MobileNetV2 + RF":       compute_metrics(y_true, y_pred_rf),
    "MobileNetV2 Fine-tuned": compute_metrics(y_true, y_pred_ft)
}

with open('model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"{'Model':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 65)
for name, m in metrics.items():
    print(f"{name:<25} {m['accuracy']:>10.2%} {m['precision']:>10.2%} {m['recall']:>10.2%} {m['f1']:>10.2%}")

print("\n=== Detailed Reports ===")
print("--- CNN ---")
print(classification_report(y_true, predicted_classes, target_names=['Normal', 'Pneumonia']))
print("--- MobileNetV2 + RF ---")
print(classification_report(y_true, y_pred_rf, target_names=['Normal', 'Pneumonia']))
print("--- MobileNetV2 Fine-tuned ---")
print(classification_report(y_true, y_pred_ft, target_names=['Normal', 'Pneumonia']))

## 10. Confusion matrices

Confusion matrices are shown for all three models.  
They reveal where each model fails: false positives (predicts pneumonia when absent) vs. false negatives (misses real pneumonia cases).  
In medical diagnosis **false negatives are more serious**: the cost of failing to detect a real pneumonia case outweighs that of a false alarm.

In [ ]:
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Pneumonia'],
                yticklabels=['Normal', 'Pneumonia'])
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

plot_confusion(y_true, predicted_classes, "Confusion Matrix - CNN")
plot_confusion(y_true, y_pred_rf,         "Confusion Matrix - MobileNetV2 + RF")
plot_confusion(y_true, y_pred_ft,         "Confusion Matrix - MobileNetV2 Fine-tuned")